# SHAP contribution study

Does SHAP improve the weighting policy beyond statistical mismatch and tails? Compare **A5**, **A5_NO_SHAP**, and **A5_SHUFFLED_SHAP**, with one shared **A0** per seed. The default three seeds require **12 GAN fits**. Normal A0–A5 runs are unchanged.

Every child uses the regular evaluation pipeline. This notebook launches a detached CLI on Linux, then displays absolute values, paired differences and spread. Three seeds provide descriptive evidence, not strong significance claims. See `docs/shap_contribution.md` for controls and limitations.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

candidates = [candidate for parent in [Path.cwd(), *Path.cwd().parents]
              for candidate in (parent, parent / 'CTAB-GAN-Plus-main')]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'xai_reweighting').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository or set PROJECT_ROOT explicitly')
print(PROJECT_ROOT)

## Plan the study before training

Choose the primary endpoint before seeing the new results. Default: scaled Wasserstein (lower is better). If utility is the main claim, deliberately select a numeric utility column from `ablation_summary.csv` instead, with direction `higher`. All secondary metrics remain available; do not switch endpoints after inspecting results.

Use a persistent results path. For WiDS, replace the config and the measured runtime estimate; do not assume MIMIC timings apply.

In [ ]:
CONFIG = PROJECT_ROOT / 'configs/mimic_ctabgan.json'
STUDY_DIR = PROJECT_ROOT / 'results/shap_contribution_mimic_ctabgan'
DEVICE = 'cuda:0'  # 'cpu' locally, or 'auto'
SEEDS = '42,43,44'
PRIMARY_METRIC = 'mean_wasserstein_scaled'
PRIMARY_DIRECTION = 'lower'
BASELINE_RUN_MINUTES = 60  # Measured full six-variant runtime, including evaluation
BUDGET_HOURS = 20
SMOKE = False
RESUME = False

command = [sys.executable, '-u', '-m', 'xai_reweighting.run_shap_contribution',
           '--config', str(CONFIG), '--output-dir', str(STUDY_DIR),
           '--device', DEVICE, '--stage', 'val', '--seeds', SEEDS,
           '--primary-metric', PRIMARY_METRIC, '--primary-direction', PRIMARY_DIRECTION,
           '--baseline-run-minutes', str(BASELINE_RUN_MINUTES), '--budget-hours', str(BUDGET_HOURS),
           '--progress', 'on']
if SMOKE:
    command.append('--smoke')
if RESUME:
    command.append('--resume')
print(shlex.join(command))

In [ ]:
# Read-only planning: no GAN fits and no results directory is created.
subprocess.run([*command, '--dry-run'], cwd=PROJECT_ROOT, check=True)

## Launch or resume (explicit opt-in)

Set `LAUNCH = True` only when ready. Linux `nohup` protects against terminal/VPN disconnection, not pod termination. Persistent output and `--resume` preserve completed seeds and variants; an interrupted fit may restart from epoch 1. Do not start two processes on the same output directory.

To resume, set `RESUME = True`, rerun the configuration cell, and launch again after checking the old process has stopped.

In [ ]:
LAUNCH = False
LOG_PATH = PROJECT_ROOT / 'logs' / (STUDY_DIR.name + ('_resume' if RESUME else '') + '.log')
PID_PATH = PROJECT_ROOT / 'logs' / (STUDY_DIR.name + '.pid')
if LAUNCH:
    if os.name != 'posix':
        raise RuntimeError('Detached launch here is for Linux/Jupyter; run the printed CLI in your local terminal on Windows')
    if PID_PATH.exists():
        previous_pid = int(PID_PATH.read_text().strip())
        try:
            os.kill(previous_pid, 0)
        except ProcessLookupError:
            pass
        else:
            raise RuntimeError(f'PID {previous_pid} still exists. Verify it before launching another process.')
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open('ab') as log:
        process = subprocess.Popen(['nohup', *command], cwd=PROJECT_ROOT,
                                   stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
                                   start_new_session=True)
    PID_PATH.write_text(str(process.pid) + '\n')
    print('Started PID', process.pid)
print('Monitor:', 'tail -f ' + shlex.quote(str(LOG_PATH)))

## Completion and primary contrasts

You can point `STUDY_DIR` at an older completed study without launching anything. Intermediate study reports include completed seeds only. Inspect coverage before interpreting spread. Positive **primary improvement** means better; raw deltas later remain in the original metric direction.

In [ ]:
STUDY_DIR = Path(STUDY_DIR).expanduser().resolve()
if not (STUDY_DIR / 'study_plan.json').exists():
    raise FileNotFoundError(f'No study found at {STUDY_DIR}')
plan = json.loads((STUDY_DIR / 'study_plan.json').read_text())
display(plan)
if (STUDY_DIR / 'manifest.json').exists():
    display(json.loads((STUDY_DIR / 'manifest.json').read_text()))
for name in ['study_run_coverage.csv', 'study_primary_summary.csv', 'study_primary_pairs.csv']:
    path = STUDY_DIR / name
    if path.exists():
        display(Markdown('### ' + name))
        display(pd.read_csv(path))
if (STUDY_DIR / 'study_primary_pairs.png').exists():
    display(Image(filename=str(STUDY_DIR / 'study_primary_pairs.png')))

## Absolute scores and spread

Full CSVs retain every available evaluation metric. Means and sample SD below are across **generator seeds**, after averaging utility repeats within each seed. Missing values are not zero. Pick an artifact, metric and context to inspect specific utility tasks, mixture fractions or features. Each plot has its own labels, so it can be captured independently.

In [ ]:
absolute = pd.read_csv(STUDY_DIR / 'study_seed_spread.csv')
paired = pd.read_csv(STUDY_DIR / 'study_paired_deltas.csv')
deltas = pd.read_csv(STUDY_DIR / 'study_delta_spread.csv')
headline = absolute[absolute.artifact == 'ablation_summary.csv']
metrics = ['mean_wasserstein_scaled', 'mean_cdf_tail_divergence', 'correlation_distance',
           'utility_mortality_pr_auc', 'utility_mortality_positive_recall',
           'privacy_median_distance_ratio', 'privacy_exact_match_rate']
display(headline[headline.metric.isin(metrics)][['variant', 'metric', 'mean', 'std', 'n', 'minimum', 'maximum', 'unavailable_seeds']])
display(Markdown('Available metric contexts (change the selectors in the next cell):'))
display(absolute[['artifact', 'metric', 'context']].drop_duplicates())

In [ ]:
ARTIFACT = 'ablation_summary.csv'  # e.g. 'utility_mixture_results.csv'
METRIC = plan['primary_metric']
contexts = absolute.loc[(absolute.artifact == ARTIFACT) & (absolute.metric == METRIC), 'context'].drop_duplicates().tolist()
CONTEXT = contexts[0] if contexts else '{}'
print('Available contexts:', contexts)
selected = absolute[(absolute.artifact == ARTIFACT) & (absolute.metric == METRIC) & (absolute.context == CONTEXT)]
selected = selected.set_index('variant').reindex(plan['variants'])
display(selected[['mean', 'std', 'n', 'minimum', 'maximum']])
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(selected))
ax.scatter(x, selected['mean'], label='Mean across generator seeds')
finite = selected['std'].notna() & selected['mean'].notna()
ax.errorbar(x[finite], selected.loc[finite, 'mean'], yerr=selected.loc[finite, 'std'], fmt='none', capsize=4, label='Sample SD')
ax.set_xticks(x, selected.index, rotation=15)
ax.set(xlabel='Variant', ylabel=METRIC, title=f'Absolute {METRIC}\n{CONTEXT}')
ax.legend()
fig.tight_layout()
plt.show()

## SHAP-specific paired differences

Raw delta = **A5 minus control**. Lower discrepancies favour negative deltas; higher utility favours positive deltas. Privacy proxies are diagnostic, not proof of privacy. Inspect regressions and missing pairs, not only improvements.

In [ ]:
contrasts = ['A5-A5_NO_SHAP', 'A5-A5_SHUFFLED_SHAP']
selected = deltas[(deltas.artifact == ARTIFACT) & (deltas.metric == METRIC) & (deltas.context == CONTEXT) & deltas.comparison.isin(contrasts)]
display(selected)
fig, ax = plt.subplots(figsize=(9, 4.5))
for index, comparison in enumerate(contrasts):
    points = paired[(paired.artifact == ARTIFACT) & (paired.metric == METRIC) & (paired.context == CONTEXT) & (paired.comparison == comparison)].dropna(subset=['delta']).sort_values('seed')
    offsets = np.linspace(-.08, .08, len(points)) if len(points) > 1 else np.zeros(len(points))
    ax.scatter(index + offsets, points.delta)
    for offset, row in zip(offsets, points.itertuples()):
        ax.annotate(str(row.seed), (index + offset, row.delta), xytext=(3, 3), textcoords='offset points')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xticks(range(len(contrasts)), contrasts)
ax.set(xlabel='Paired comparison', ylabel=f'{METRIC}: A5 minus control', title=f'Generator-seed differences\n{CONTEXT}')
fig.tight_layout()
plt.show()

## Sampling concentration and complete evaluation inventory

A shuffled importance vector can produce more or less concentrated row weights. Compare ESS, weight SD and cap fractions before attributing a difference exclusively to better feature ordering. ESS here describes augmentation draw probabilities, not the privacy or number of independent patients.

In [ ]:
weights = pd.read_csv(STUDY_DIR / 'study_weight_diagnostics.csv')
display(pd.read_csv(STUDY_DIR / 'study_mechanism_diagnostics.csv'))
display(weights[weights.metric.isin(['mean', 'std', 'fraction_capped', 'sampling_ess', 'sampling_ess_fraction', 'maximum_sampling_probability'])])
display(pd.read_csv(STUDY_DIR / 'study_artifact_coverage.csv'))
display(json.loads((STUDY_DIR / 'study_interpretation.json').read_text()))

## Refresh reports without training

The original config, seeds, device, endpoint and code must still match. This only aggregates completed child artifacts; it does not rerun evaluations. Use the regular `run_evaluation --run-dir .../seed42` CLI first if needed.

In [ ]:
REFRESH_REPORTS = False
if REFRESH_REPORTS:
    subprocess.run([*command, '--summarize-only'], cwd=PROJECT_ROOT, check=True)